# Epic 3 -- CycleGAN Cross-Modality Translation Overview

This notebook introduces the CycleGAN approach used in Epic 3 for **cross-modality
image translation** between AOI (Automated Optical Inspection) and USM (Ultrasonic
Microscopy) imaging modalities.

The core problem: we have labeled data for AOI images but **no labels for USM**.
Manually labeling USM images is expensive and time-consuming.  CycleGAN solves
this by learning to translate AOI images into realistic USM images, so we can
transfer the AOI labels to the USM domain and train downstream models without
any USM annotations.

## What you will learn

1. The CycleGAN architecture: two generators, two discriminators, cycle consistency
2. Why unpaired image translation works for cross-modality transfer
3. How to create a `CycleGANModel` and run a forward pass on random data
4. The loss components that drive training

## Prerequisites

- Python 3.12, PyTorch
- Install: `uv pip install -e ".[epic3]"`

---
## 1. The Problem: Labeled AOI, Unlabeled USM

We have two imaging modalities used in manufacturing quality control:

| Modality | Images | Labels | Strength |
|----------|--------|--------|----------|
| **AOI** (Automated Optical Inspection) | 2000+ | Full defect masks | Fast, cheap, surface-level |
| **USM** (Ultrasonic Microscopy) | 1500+ | **None** | Sub-surface defects, high detail |

AOI captures surface defects quickly, but USM reveals internal structure that AOI
cannot see.  We want to train a defect segmentation model for USM images, but
labeling USM data requires expensive expert annotation.

**Solution:** Translate AOI images into USM-style images using CycleGAN, then
reuse the AOI labels on the translated images to train a USM segmentation model.

---
## 2. Architecture Overview

CycleGAN learns **two** translation directions simultaneously using unpaired data:

```
                    Generator G_AB                    Generator G_BA
        AOI image ──────────────────> Fake USM    USM image ──────────────────> Fake AOI
            |                            |            |                            |
            |                            v            |                            v
            |                    Discriminator D_B    |                    Discriminator D_A
            |                    (real vs fake USM)   |                    (real vs fake AOI)
            |                                         |                                    
            |          Cycle consistency               |          Cycle consistency         
            |          ┌─────────────────────┐        |          ┌─────────────────────┐   
            +--------->│ G_BA(G_AB(AOI))     │        +--------->│ G_AB(G_BA(USM))     │   
                       │   should == AOI     │                   │   should == USM     │   
                       └─────────────────────┘                   └─────────────────────┘   
```

### Key components

| Component | Purpose |
|-----------|---------|
| **Generator G_AB** (AOI -> USM) | Translate AOI images to look like USM |
| **Generator G_BA** (USM -> AOI) | Translate USM images to look like AOI |
| **Discriminator D_A** | Distinguish real AOI from fake AOI (generated by G_BA) |
| **Discriminator D_B** | Distinguish real USM from fake USM (generated by G_AB) |

### Loss components

| Loss | Formula | Purpose |
|------|---------|---------|
| **Adversarial** | LSGAN (least-squares) | Make generated images look realistic |
| **Cycle consistency** | L1(G_BA(G_AB(x)) - x) | Preserve content through round-trip |
| **Identity** | L1(G_AB(usm) - usm) | Preserve color/tone when input is already target domain |
| **Defect preservation** | Dice on defect masks | Keep defect regions intact (Epic 3.3) |

---
## 3. Quick Demo: Creating and Running a CycleGANModel

Let's instantiate the full CycleGAN model and run a forward pass on random data
to see the input/output shapes.

In [ ]:
import torch

from udm_epic3.models.generator import ResnetGenerator
from udm_epic3.models.discriminator import PatchDiscriminator
from udm_epic3.models.cyclegan import CycleGANModel

In [ ]:
# Create a CycleGAN model with default settings.
model = CycleGANModel(
    in_channels=3,
    out_channels=3,
    n_residual_blocks=9,      # 9 ResNet blocks in each generator
    n_discriminator_layers=3,  # PatchGAN with 3 conv layers
    lambda_cycle=10.0,         # cycle consistency weight
    lambda_identity=0.5,       # identity loss weight
    lambda_defect=0.0,         # defect preservation (disabled for baseline)
)

total_params = sum(p.numel() for p in model.parameters())
print(f"CycleGAN model created.  Total parameters: {total_params:,}")
print(f"  Generator G_AB params:    {sum(p.numel() for p in model.G_AB.parameters()):,}")
print(f"  Generator G_BA params:    {sum(p.numel() for p in model.G_BA.parameters()):,}")
print(f"  Discriminator D_A params: {sum(p.numel() for p in model.D_A.parameters()):,}")
print(f"  Discriminator D_B params: {sum(p.numel() for p in model.D_B.parameters()):,}")

In [ ]:
# Simulate a batch of AOI and USM images (unpaired)
real_A = torch.randn(2, 3, 256, 256)  # AOI images
real_B = torch.randn(2, 3, 256, 256)  # USM images

# Forward pass: generates fakes and reconstructions
model.eval()
with torch.no_grad():
    outputs = model(real_A, real_B)

print(f"Input AOI (real_A) shape:       {real_A.shape}")
print(f"Fake USM (fake_B) shape:        {outputs['fake_B'].shape}")
print(f"Reconstructed AOI (rec_A) shape: {outputs['rec_A'].shape}")
print()
print(f"Input USM (real_B) shape:       {real_B.shape}")
print(f"Fake AOI (fake_A) shape:        {outputs['fake_A'].shape}")
print(f"Reconstructed USM (rec_B) shape: {outputs['rec_B'].shape}")

**What the outputs mean:**

- `fake_B`: AOI translated to USM style (this is what we ultimately want)
- `rec_A`: `fake_B` translated back to AOI -- should match `real_A` (cycle consistency)
- `fake_A`: USM translated to AOI style
- `rec_B`: `fake_A` translated back to USM -- should match `real_B` (cycle consistency)

The generators output images in the same spatial resolution as the input.  The
ResNet-based architecture uses an encoder-residual-decoder structure that preserves
spatial dimensions through residual blocks.

---
## 4. Generator Architecture: ResNet Generator

Each generator follows the architecture from Johnson et al. (2016):

```
Input [B, 3, H, W]
    |
    v
Reflection padding + 7x7 conv + InstanceNorm + ReLU
    |
    v
Downsample (2x stride-2 convolutions)  -->  [B, 256, H/4, W/4]
    |
    v
N Residual Blocks (default N=9)        -->  [B, 256, H/4, W/4]
    |
    v
Upsample (2x transposed convolutions)  -->  [B, 64, H, W]
    |
    v
Reflection padding + 7x7 conv + Tanh   -->  [B, 3, H, W]
```

Key design choices:
- **Instance normalization** instead of batch norm (better for style transfer)
- **Reflection padding** to avoid border artifacts
- **Tanh** output to produce images in [-1, 1] range

In [ ]:
# Inspect the generator architecture
gen = ResnetGenerator(in_channels=3, out_channels=3, n_residual_blocks=9)
print(f"Generator parameters: {sum(p.numel() for p in gen.parameters()):,}")
print(f"\nGenerator structure (summary):")
print(f"  - Initial conv: 7x7, 3 -> 64 channels")
print(f"  - Downsample 1: 3x3, 64 -> 128 channels")
print(f"  - Downsample 2: 3x3, 128 -> 256 channels")
print(f"  - Residual blocks: 9 x (256 -> 256)")
print(f"  - Upsample 1: 3x3, 256 -> 128 channels")
print(f"  - Upsample 2: 3x3, 128 -> 64 channels")
print(f"  - Final conv: 7x7, 64 -> 3 channels")

---
## 5. Discriminator Architecture: PatchGAN

The discriminator is a **PatchGAN** (Isola et al., 2017).  Instead of classifying
the entire image as real or fake, it produces a grid of real/fake predictions,
one for each overlapping patch.  This encourages the generator to produce
realistic local texture.

```
Input [B, 3, H, W]
    |
    v
4x4 conv, stride 2 (no norm)  -->  [B, 64, H/2, W/2]
    |
    v
4x4 conv, stride 2 + InstanceNorm  -->  [B, 128, H/4, W/4]
    |
    v
4x4 conv, stride 2 + InstanceNorm  -->  [B, 256, H/8, W/8]
    |
    v
4x4 conv, stride 1 + InstanceNorm  -->  [B, 512, H/8-1, W/8-1]
    |
    v
4x4 conv, stride 1 (1 channel)    -->  [B, 1, H/8-2, W/8-2]
```

Each value in the output grid classifies a ~70x70 receptive field patch.

In [ ]:
# Inspect the discriminator
disc = PatchDiscriminator(in_channels=3, n_layers=3)
dummy = torch.randn(1, 3, 256, 256)
with torch.no_grad():
    patch_out = disc(dummy)

print(f"Discriminator parameters: {sum(p.numel() for p in disc.parameters()):,}")
print(f"Input shape:  {dummy.shape}")
print(f"Output shape: {patch_out.shape}")
print(f"\nEach value in the {patch_out.shape[2]}x{patch_out.shape[3]} grid is a")
print(f"real/fake score for one overlapping image patch.")

---
## 6. Project Structure

| Module | Description |
|--------|------------|
| `udm_epic3.models.generator` | `ResnetGenerator` -- encoder-residual-decoder generator |
| `udm_epic3.models.discriminator` | `PatchDiscriminator` -- PatchGAN discriminator |
| `udm_epic3.models.cyclegan` | `CycleGANModel` -- composes generators + discriminators |
| `udm_epic3.models.losses` | `adversarial_loss_lsgan`, `cycle_consistency_loss`, `identity_loss`, `defect_preservation_loss` |
| `udm_epic3.data.unpaired_dataset` | `UnpairedDataset`, `PairedDataset` |
| `udm_epic3.data.image_pool` | `ImagePool` -- stores previously generated images for stable training |
| `udm_epic3.training.train_cyclegan` | `train_cyclegan_from_yaml` |
| `udm_epic3.translation.translate` | `translate_single`, `translate_directory` |
| `udm_epic3.evaluation.quality_metrics` | `compute_ssim`, `compute_defect_dice`, `evaluate_translation` |

## CLI overview

```bash
# Prepare data (US 3.1)
udm-epic3 prepare --config configs/epic3_cyclegan.yaml

# Train baseline CycleGAN (US 3.2)
udm-epic3 train --config configs/epic3_cyclegan.yaml

# Evaluate translation quality (US 3.4)
udm-epic3 evaluate --config configs/epic3_cyclegan.yaml

# Translate AOI images to USM (US 3.5)
udm-epic3 translate --input data/aoi --output data/translated_usm

# Validate on multiple sites (US 3.6)
udm-epic3 validate --config configs/epic3_cyclegan.yaml
```

## Next steps

| Notebook | Topic |
|----------|------|
| `epic3_01_data_prep.ipynb` | Multi-modal data collection and preparation |
| `epic3_02_training.ipynb` | Baseline CycleGAN training |
| `epic3_03_defect_aware.ipynb` | Defect-aware enhancement |
| `epic3_04_evaluation.ipynb` | Translation quality evaluation |
| `epic3_05_downstream.ipynb` | Downstream model training |
| `epic3_06_multisite.ipynb` | Multi-site generalization |